In [11]:
# Load Packages
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os

In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset 

In [8]:
import torchvision.models as models

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Load fresh data
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),   # greyscale but 3 channels
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],    # ImageNet mean
        std=[0.229, 0.224, 0.225]      # ImageNet std
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    root=r"C:\Users\Sandeep\Desktop\Projects\data\train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=r"C:\Users\Sandeep\Desktop\Projects\data\train",
    transform=val_transform
)

# generate indices manually and slice directly
indices    = list(range(len(train_dataset)))

# shuffle indices first so split is random
import random
random.seed(42)                        # reproducible split every run
random.shuffle(indices)

train_size    = int(0.8 * len(indices))
train_indices = indices[:train_size]
val_indices   = indices[train_size:]

# now Subset gets clean plain lists, not Subset objects
train_subset = Subset(train_dataset, train_indices)
val_subset   = Subset(val_dataset,   val_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_subset,   batch_size=32, shuffle=False)

# verify
print(f"Train size: {len(train_subset)}")
print(f"Val size:   {len(val_subset)}")

images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")

Train size: 22618
Val size:   5655
Batch shape: torch.Size([32, 3, 224, 224])


In [19]:
# load pretrained ResNet18
resnet = models.resnet18(weights="IMAGENET1K_V1")

# freeze all layers - we don't want to touch pretrained weights yet
for param in resnet.parameters():
    param.requires_grad = False

# replace the final layer only - this is the only part we train
# resnet18 final layer outputs 512 -> 1000 (imagenet classes)
# we swap it to 512 -> 6 (our emotions)
resnet.fc = nn.Linear(resnet.fc.in_features, 6)

# move to GPU
model = resnet.to(device)

# sanity check
dummy  = torch.randn(32, 3, 224, 224).to(device)
output = model(dummy)
print(output.shape)   # should be torch.Size([32, 6])

# confirm only final layer is training
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")   # should be much smaller

torch.Size([32, 6])
Total params:     11,179,590
Trainable params: 3,078


In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)

num_epochs       = 20     # less epochs needed, pretrained weights converge fast
patience         = 5
best_val_loss    = float("inf")
patience_counter = 0

for epoch in range(num_epochs):

    # ---- train ----
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    # ---- val ----
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_resnet18.pth")
        print(f"  ✓ Best model saved (val loss: {val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_resnet18.pth"))
print("Best model loaded")

Epoch [1/20]  Train Loss: 1.5921  Train Acc: 35.42%  |  Val Loss: 1.4906  Val Acc: 40.62%
  ✓ Best model saved (val loss: 1.4906)
Epoch [2/20]  Train Loss: 1.4959  Train Acc: 40.60%  |  Val Loss: 1.4645  Val Acc: 42.58%
  ✓ Best model saved (val loss: 1.4645)
Epoch [3/20]  Train Loss: 1.4760  Train Acc: 41.88%  |  Val Loss: 1.4643  Val Acc: 42.26%
  ✓ Best model saved (val loss: 1.4643)
Epoch [4/20]  Train Loss: 1.4704  Train Acc: 42.09%  |  Val Loss: 1.4766  Val Acc: 42.33%
  No improvement (1/5)
Epoch [5/20]  Train Loss: 1.4641  Train Acc: 42.24%  |  Val Loss: 1.4435  Val Acc: 43.57%
  ✓ Best model saved (val loss: 1.4435)
Epoch [6/20]  Train Loss: 1.4515  Train Acc: 43.09%  |  Val Loss: 1.4607  Val Acc: 41.68%
  No improvement (1/5)
Epoch [7/20]  Train Loss: 1.4476  Train Acc: 42.76%  |  Val Loss: 1.4956  Val Acc: 42.32%
  No improvement (2/5)
Epoch [8/20]  Train Loss: 1.4496  Train Acc: 42.98%  |  Val Loss: 1.6078  Val Acc: 38.71%
  No improvement (3/5)
Epoch [9/20]  Train Loss: 1.

In [26]:
# --- Trial 2 ---
# reload fresh model
resnet = models.resnet18(weights="IMAGENET1K_V1")

# unfreeze only the last residual block (layer4) + final layer
# freeze everything before that
for name, param in resnet.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True    # training these
    else:
        param.requires_grad = False   # freezing these

# replace final layer
resnet.fc = nn.Linear(resnet.fc.in_features, 6)
model = resnet.to(device)

# confirm
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")   

Total params:     11,179,590
Trainable params: 8,396,806


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.0001)   # 10x smaller than before

num_epochs       = 20     # less epochs needed, pretrained weights converge fast
patience         = 7
best_val_acc     = 0.0
patience_counter = 0

for epoch in range(num_epochs):

    # ---- train ----
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    # ---- val ----
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    # early stopping tracks val accuracy
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_resnet18.pth")
        print(f"  ✓ Best model saved (val acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_resnet18.pth"))
print("Best model loaded")

Epoch [1/20]  Train Loss: 1.2248  Train Acc: 52.41%  |  Val Loss: 1.0570  Val Acc: 59.50%
  ✓ Best model saved (val acc: 59.50%)
Epoch [2/20]  Train Loss: 1.0197  Train Acc: 61.23%  |  Val Loss: 0.9944  Val Acc: 62.00%
  ✓ Best model saved (val acc: 62.00%)
Epoch [3/20]  Train Loss: 0.9124  Train Acc: 65.50%  |  Val Loss: 1.0022  Val Acc: 62.81%
  ✓ Best model saved (val acc: 62.81%)
Epoch [4/20]  Train Loss: 0.8442  Train Acc: 68.42%  |  Val Loss: 1.0262  Val Acc: 62.94%
  ✓ Best model saved (val acc: 62.94%)
Epoch [5/20]  Train Loss: 0.7631  Train Acc: 71.60%  |  Val Loss: 1.0344  Val Acc: 63.43%
  ✓ Best model saved (val acc: 63.43%)
Epoch [6/20]  Train Loss: 0.6841  Train Acc: 74.88%  |  Val Loss: 1.0404  Val Acc: 64.01%
  ✓ Best model saved (val acc: 64.01%)
Epoch [7/20]  Train Loss: 0.6164  Train Acc: 77.12%  |  Val Loss: 1.0902  Val Acc: 63.54%
  No improvement (1/7)
Epoch [8/20]  Train Loss: 0.5589  Train Acc: 79.80%  |  Val Loss: 1.0994  Val Acc: 64.17%
  ✓ Best model saved (v

KeyboardInterrupt: 

In [28]:
# ==== Trial 3 ====
# fresh model againnnnnn
resnet = models.resnet18(weights="IMAGENET1K_V1")

for name, param in resnet.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

resnet.fc = nn.Linear(resnet.fc.in_features, 6)
model     = resnet.to(device)

# discriminative learning rates
optimizer = Adam([
    {"params": resnet.layer4.parameters(), "lr": 0.00001},  # 10x smaller
    {"params": resnet.fc.parameters(),     "lr": 0.0001}    # normal
])

criterion        = nn.CrossEntropyLoss()
num_epochs       = 20
patience         = 10
best_val_acc     = 0.0
patience_counter = 0

In [29]:
for epoch in range(num_epochs):

    # ---- train ----
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    # ---- val ----
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    # early stopping tracks val accuracy
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_resnet18.pth")
        print(f"  ✓ Best model saved (val acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_resnet18.pth"))
print("Best model loaded")

Epoch [1/20]  Train Loss: 1.4749  Train Acc: 41.46%  |  Val Loss: 1.2971  Val Acc: 49.73%
  ✓ Best model saved (val acc: 49.73%)
Epoch [2/20]  Train Loss: 1.2336  Train Acc: 52.29%  |  Val Loss: 1.1969  Val Acc: 54.20%
  ✓ Best model saved (val acc: 54.20%)
Epoch [3/20]  Train Loss: 1.1512  Train Acc: 55.71%  |  Val Loss: 1.1323  Val Acc: 56.87%
  ✓ Best model saved (val acc: 56.87%)
Epoch [4/20]  Train Loss: 1.0919  Train Acc: 58.09%  |  Val Loss: 1.1014  Val Acc: 58.05%
  ✓ Best model saved (val acc: 58.05%)
Epoch [5/20]  Train Loss: 1.0504  Train Acc: 60.16%  |  Val Loss: 1.0762  Val Acc: 59.12%
  ✓ Best model saved (val acc: 59.12%)
Epoch [6/20]  Train Loss: 1.0108  Train Acc: 61.85%  |  Val Loss: 1.0586  Val Acc: 60.05%
  ✓ Best model saved (val acc: 60.05%)
Epoch [7/20]  Train Loss: 0.9770  Train Acc: 63.29%  |  Val Loss: 1.0431  Val Acc: 60.46%
  ✓ Best model saved (val acc: 60.46%)
Epoch [8/20]  Train Loss: 0.9492  Train Acc: 64.18%  |  Val Loss: 1.0384  Val Acc: 60.67%
  ✓ Bes

In [30]:
# Trial 4 — unfreeze layer3 + layer4 + fc
resnet = models.resnet18(weights="IMAGENET1K_V1")

for name, param in resnet.named_parameters():
    if "layer3" in name or "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

resnet.fc = nn.Linear(resnet.fc.in_features, 6)
model     = resnet.to(device)

# discriminative learning rates
optimizer = Adam([
    {"params": resnet.layer3.parameters(), "lr": 0.000001},  # very careful
    {"params": resnet.layer4.parameters(), "lr": 0.00001},
    {"params": resnet.fc.parameters(),     "lr": 0.0001}
])

criterion        = nn.CrossEntropyLoss()
num_epochs       = 20
patience         = 10
best_val_acc     = 0.0
patience_counter = 0

In [31]:
for epoch in range(num_epochs):

    # ---- train ----
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    # ---- val ----
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    # early stopping tracks val accuracy
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_resnet18.pth")
        print(f"  ✓ Best model saved (val acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_resnet18.pth"))
print("Best model loaded")

Epoch [1/20]  Train Loss: 1.4558  Train Acc: 42.44%  |  Val Loss: 1.2720  Val Acc: 51.09%
  ✓ Best model saved (val acc: 51.09%)
Epoch [2/20]  Train Loss: 1.2157  Train Acc: 52.80%  |  Val Loss: 1.1722  Val Acc: 55.08%
  ✓ Best model saved (val acc: 55.08%)
Epoch [3/20]  Train Loss: 1.1283  Train Acc: 56.95%  |  Val Loss: 1.1107  Val Acc: 57.75%
  ✓ Best model saved (val acc: 57.75%)
Epoch [4/20]  Train Loss: 1.0755  Train Acc: 58.79%  |  Val Loss: 1.0806  Val Acc: 59.43%
  ✓ Best model saved (val acc: 59.43%)
Epoch [5/20]  Train Loss: 1.0245  Train Acc: 61.03%  |  Val Loss: 1.0563  Val Acc: 60.64%
  ✓ Best model saved (val acc: 60.64%)
Epoch [6/20]  Train Loss: 0.9873  Train Acc: 62.40%  |  Val Loss: 1.0380  Val Acc: 61.10%
  ✓ Best model saved (val acc: 61.10%)
Epoch [7/20]  Train Loss: 0.9534  Train Acc: 64.19%  |  Val Loss: 1.0250  Val Acc: 61.54%
  ✓ Best model saved (val acc: 61.54%)
Epoch [8/20]  Train Loss: 0.9175  Train Acc: 65.34%  |  Val Loss: 1.0179  Val Acc: 62.19%
  ✓ Bes